<a href="https://colab.research.google.com/github/zith008/ai-book-exercises/blob/main/notebooks/Chapter_09_Vector_Databases_SOLUTIONS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open this notebook in Colab"/></a>

# ⚠️ Solutions — Chapter 9

The worked key for *Chapter 9 — Vector Databases*. Every cell is complete and the notebook
runs top to bottom.

Check your own attempt against it, or use it to unblock one line. Reading a solution you
have not attempted feels like learning and is not.

# Chapter 9 — Vector Databases
### Exercise notebook · *Developer Guide: Building with AI*


A vector database finds the nearest stored vectors to a query. Everything interesting is in
what it trades away to do that quickly, and in what your code assumes about it.

Three exercises, each about a failure that is **quiet** — no error, no crash, just wrong
results that look right: an index that misses answers, a vendor swap that changes
behaviour, and a permission filter that only covers half the search.


| Exercise | Book section | What you write |
|---|---|---|
| 1 | 9.1 | What an approximate index costs you |
| 2 | 9.2 | One contract, any database |
| 3 | 9.3 | A filter that leaks |

**About 35 minutes.** **No API key needed.** A stand-in model is built in, so everything here runs free and offline — exactly as the book's own scripts do.

**How this works:** run the cells in order. Each exercise has `# TODO` comments and ends
with `raise NotImplementedError` — write your code, delete that line, run the cell. Then
run the check cell underneath: it prints ✅, or it tells you what is wrong. Stuck for ten
minutes? Run the `hint(...)` cell. Nobody is marking you.

In [ ]:
#@title Setup — run this cell first (double-click to read the code)
import sys, time

_RESULTS = {}   # exercise label -> "pass" | "fail" | "todo" | "error"

def check(label):
    """Decorator that runs a block of assertions and reports the result.

    Use it like this:

        @check("1.1  single neuron")
        def _():
            assert my_thing() == 42, "a message the reader can act on"
    """
    def run(fn):
        started = time.time()
        try:
            fn()
        except NotImplementedError as err:
            _RESULTS[label] = "todo"
            reason = str(err) or ("not attempted yet (write your code, then delete "
                                  "the `raise NotImplementedError`)")
            print(f"⬜  {label}  —  {reason}")
        except AssertionError as err:
            _RESULTS[label] = "fail"
            print(f"❌  {label}\n     {err}")
        except Exception as err:                     # noqa: BLE001 - readers see the type
            _RESULTS[label] = "error"
            print(f"⚠️   {label}  —  {type(err).__name__}: {err}")
            print("     The code crashed before the check could finish. "
                  "Scroll up to the traceback or re-run your cell on its own.")
        else:
            _RESULTS[label] = "pass"
            print(f"✅  {label}   ({time.time() - started:.1f}s)")
        return fn
    return run

def progress():
    """Print a summary of every check run so far in this session."""
    if not _RESULTS:
        print("No checks have run yet. Run the exercise cells above.")
        return
    icon = {"pass": "✅", "fail": "❌", "todo": "⬜", "error": "⚠️ "}
    done = sum(1 for v in _RESULTS.values() if v == "pass")
    width = max(len(k) for k in _RESULTS)
    print(f"{done}/{len(_RESULTS)} checks passing\n" + "-" * (width + 6))
    for label, state in _RESULTS.items():
        print(f"{icon[state]}  {label}")
    if done < len(_RESULTS):
        print("\nAnything not green is a hint, not a grade. "
              "Read the message, change one thing, re-run the cell.")

def need(*names):
    """Assert that the exercise cell above has been run.

    Gives a readable message instead of a bare NameError when a reader runs a
    check cell before the exercise it belongs to.
    """
    missing = [n for n in names if n not in globals()]
    if missing:
        raise NotImplementedError(
            f"{', '.join(missing)} is not defined yet — run the exercise cell "
            "directly above this one (and make sure it finished without an error)."
        )

def need_data(ok, what="the exercise above"):
    """Like need(), but for state that exists and is empty until the exercise runs."""
    if not ok:
        raise NotImplementedError(
            f"nothing to check yet — run {what} and make sure it finished without "
            "an error, then run this cell again."
        )

def hint(*lines):
    """Print a hint. Readers call this only when they want it."""
    print("\n".join(lines))

print(f"Python {sys.version.split()[0]} — setup complete. "
      "Run the cells in order; call progress() at any time.")

---
## Setup — the code you are given

Read this, then run it. You will use these pieces in the exercises; you do not need to
write them.

In [ ]:
import re, hashlib
import numpy as np

EMBED_DIM = 16384
STOPWORDS = {"a", "an", "the", "is", "are", "to", "of", "and", "in", "on", "for",
             "it", "that", "this", "with", "as", "by", "be", "or", "from", "at",
             "you", "your", "we", "i", "what", "how", "do", "does", "can", "if"}


def _words(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS]


def embed(text, model="stand-in-v1", dim=EMBED_DIM):
    """Turn text into a unit vector: one slot per word of a large vocabulary.

    Each word is hashed to its own slot and adds 1 there, then the vector is scaled to
    length 1. Texts sharing words point in similar directions; texts sharing nothing are
    at right angles, which is a distance of exactly 1.

    A real embedder is a neural network, and understands that "refund window" and
    "return within 30 days" mean the same thing. This one only sees shared words. The
    mechanics of searching, cutoffs and recall are identical; the matching is cruder.
    """
    vector = np.zeros(dim)
    salt = model.encode()
    for word in _words(text):
        position = int(hashlib.md5(salt + word.encode()).hexdigest(), 16) % dim
        vector[position] += 1.0
    length = np.linalg.norm(vector)
    return vector / length if length else vector


def distance(a, b):
    """Cosine distance: 0.0 is identical direction, 1.0 is nothing in common."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    size = np.linalg.norm(a) * np.linalg.norm(b)
    return 1.0 if size == 0 else float(1 - np.dot(a, b) / size)

import random

# ---- A document set with owners and visibility ------------------------------
DOCS = [
    # id, text, team, visibility
    ("d1", "Returns policy: items may be returned within 30 days for a full refund.",
     "support", "public"),
    ("d2", "Shipping: express delivery costs 9.99 GBP and arrives next working day.",
     "support", "public"),
    ("d3", "Warranty: electronics carry a 24 month warranty for manufacturing faults.",
     "support", "public"),
    ("d4", "Salary bands for the support team are reviewed every March.",
     "hr", "confidential"),
    ("d5", "Disciplinary process: a written warning precedes any dismissal.",
     "hr", "confidential"),
    ("d6", "Refund abuse: accounts with repeated refund claims are flagged for review.",
     "support", "internal"),
    ("d7", "Payment gateway credentials rotate every 90 days.",
     "platform", "confidential"),
    ("d8", "Account closure removes order history permanently and cannot be undone.",
     "support", "public"),
]

RECORDS = [{"id": i, "text": t, "team": team, "visibility": v}
           for i, t, team, v in DOCS]
VECTORS = {r["id"]: embed(r["text"]) for r in RECORDS}


# ---- An exact store: compares against everything, always right, never fast --
class ExactStore:
    def search(self, query, top_k=3, records=None):
        records = RECORDS if records is None else records
        q = embed(query)
        scored = [(distance(q, VECTORS[r["id"]]), r["id"]) for r in records]
        scored.sort(key=lambda pair: pair[0])
        return [doc_id for _, doc_id in scored[:top_k]]


# ---- An "approximate" store: the kind of index a real database uses ---------
# It groups vectors into buckets and searches only the nearest few buckets.
# That is the speed-for-accuracy trade every approximate index makes.
class ApproximateStore:
    def __init__(self, buckets=4, probe=1, seed=0):
        self.probe = probe                   # how many buckets to look inside
        rng = random.Random(seed)
        ids = [r["id"] for r in RECORDS]
        rng.shuffle(ids)
        self.buckets = [ids[i::buckets] for i in range(buckets)]
        self.centroids = [
            sum((VECTORS[i] for i in bucket), start=np.zeros(EMBED_DIM)) / max(len(bucket), 1)
            for bucket in self.buckets
        ]

    def search(self, query, top_k=3):
        q = embed(query)
        order = sorted(range(len(self.buckets)),
                       key=lambda b: distance(q, self.centroids[b]))
        candidates = [i for b in order[:self.probe] for i in self.buckets[b]]
        scored = sorted((distance(q, VECTORS[i]), i) for i in candidates)
        return [doc_id for _, doc_id in scored[:top_k]]


QUERIES = ["how long do I have to return an item",
           "express delivery cost",
           "warranty on electronics",
           "closing my account",
           "refund abuse flagging"]

print(f"{len(RECORDS)} documents indexed; {len(QUERIES)} test queries")

---
## Exercise 1 — What an approximate index costs you  ·  *book §9.1*


A vector database does one thing: given a query vector, find the nearest stored vectors.

The honest way is to compare against **every** vector. That is exact, and slow — fine for
10,000 documents, hopeless for 10 million.

So real databases use an **approximate index**. They group vectors and search only the
promising groups. It is dramatically faster, and it sometimes **misses the right answer** —
not because of a bug, but by design.

The measure of that miss is **recall**: of the results exact search would have given you,
what fraction did the approximate index find?

$$\text{recall@k} = \frac{|\text{approximate top-k} \cap \text{exact top-k}|}{k}$$

Recall of 1.0 means you lost nothing. 0.6 means four in ten of your best results never
reached the user — and nothing anywhere will warn you, because the search still returned
results and they still looked plausible.

> Measure recall against exact search on your own data before you trust anyone's defaults.

In [ ]:
exact = ExactStore()


def recall_at_k(approximate_store, queries, k=3):
    """The average fraction of exact results the approximate store also found."""
    scores = []
    for query in queries:
        truth = set(exact.search(query, top_k=k))
        found = set(approximate_store.search(query, top_k=k))
        scores.append(len(found & truth) / k)
    return sum(scores) / len(scores)


print(f"{'probe':>6} {'recall@3':>10}   (how many buckets each search looks inside)")
for probe in (1, 2, 3, 4):
    store = ApproximateStore(buckets=4, probe=probe)
    print(f"{probe:>6} {recall_at_k(store, QUERIES):>10.2f}")

one_bucket = ApproximateStore(buckets=4, probe=1)
print("\nexample query:", QUERIES[0])
print("  exact says:      ", exact.search(QUERIES[0]))
print("  approximate says:", one_bucket.search(QUERIES[0]))

In [ ]:
@check("9.1  recall is measured correctly")
def _():
    need("recall_at_k", "exact")

    class Perfect:
        def search(self, query, top_k=3):
            return exact.search(query, top_k=top_k)

    class Useless:
        def search(self, query, top_k=3):
            return []

    assert abs(recall_at_k(Perfect(), QUERIES) - 1.0) < 1e-9, (
        "a store that returns exactly the exact results must score 1.0; you got "
        f"{recall_at_k(Perfect(), QUERIES):.3f}"
    )
    assert recall_at_k(Useless(), QUERIES) == 0.0, \
        "a store that returns nothing must score 0.0"

    class Half:
        def search(self, query, top_k=3):
            return exact.search(query, top_k=top_k)[:1] + ["nonsense", "garbage"]

    assert abs(recall_at_k(Half(), QUERIES, k=3) - 1 / 3) < 1e-9, (
        "a store finding 1 of the 3 right answers should score 0.33 — divide the "
        f"overlap by k, not by the number of results. You got "
        f"{recall_at_k(Half(), QUERIES, k=3):.3f}"
    )


@check("9.1  searching more buckets raises recall")
def _():
    need("recall_at_k")
    narrow = recall_at_k(ApproximateStore(buckets=4, probe=1), QUERIES)
    wide = recall_at_k(ApproximateStore(buckets=4, probe=4), QUERIES)
    assert wide >= narrow, \
        f"probing every bucket ({wide:.2f}) should be at least as good as one ({narrow:.2f})"
    assert wide > 0.99, (
        f"probing ALL 4 buckets gave recall {wide:.2f}. Looking everywhere is the same "
        "as exact search, so it should be 1.0 — if it is not, the recall calculation "
        "is off."
    )
    assert narrow < 0.99, (
        f"probing one bucket of four gave recall {narrow:.2f}. It should lose something: "
        "that loss is the price of the speed, and the reason to measure it."
    )
    print(f"     recall {narrow:.2f} with 1 bucket, {wide:.2f} with all 4 "
          "— that gap is the trade")

In [ ]:
hint('recall_at_k:',
     '  scores = []',
     '  for query in queries:',
     '      truth = set(exact.search(query, top_k=k))',
     '      found = set(approximate_store.search(query, top_k=k))',
     '      scores.append(len(found & truth) / k)',
     '  return sum(scores) / len(scores)',
     '',
     'Divide by k, not by len(found). A store returning one correct result out of',
     'one has not achieved perfect recall — it missed k-1 others.')

**What to notice.** The approximate search returned confident, sensible-looking results and
was missing the best match. There was no error, no warning, no slow query to notice.

This is the quiet failure mode of vector databases, and the only defence is the number you
just computed — measured on **your** documents and **your** queries, because recall depends
on both.

---
## Exercise 2 — One contract, any database  ·  *book §9.2*


You will change vector database at least once: pricing changes, scale changes, or you move
from a local prototype to a hosted service.

That is cheap if every store sits behind **one small contract**, and expensive if your
application code calls each vendor's API directly. The difference is where you put the
translation.

Here the contract is three methods:

```python
add(records)          # store documents and their vectors
query(text, top_k, where=None)   # search, optionally filtered
count()               # how many documents are stored
```

The awkward part is always **filters**, because every vendor spells them differently:

| Our contract | Store A | Store B |
|---|---|---|
| `{"team": "support"}` | `{"team": {"$eq": "support"}}` | `"team == 'support'"` |

Translate in **one place** — inside the adapter. Then the same tests run against both
stores, and a vendor's odd syntax never leaks into your application.

In [ ]:
# ---- Two "vendors", with deliberately different filter syntax --------------
class VendorA:
    """Filters look like {"team": {"$eq": "support"}}."""
    def __init__(self):
        self.items = {}
    def upsert(self, items):
        for item in items:
            self.items[item["id"]] = item
    def knn(self, vector, k, filter=None):
        matches = list(self.items.values())
        if filter:
            for field, condition in filter.items():
                wanted = condition["$eq"]
                matches = [m for m in matches if m["metadata"].get(field) == wanted]
        scored = sorted((distance(vector, m["values"]), m["id"]) for m in matches)
        return [doc_id for _, doc_id in scored[:k]]


class VendorB:
    """Filters are a string expression, e.g. "team == 'support'"."""
    def __init__(self):
        self.rows = []
    def insert(self, rows):
        self.rows.extend(rows)
    def nearest(self, vector, limit, expression=None):
        rows = self.rows
        if expression:
            field, _, raw = expression.split(" ", 2)
            wanted = raw.strip().strip("'")
            rows = [r for r in rows if r["meta"].get(field) == wanted]
        scored = sorted((distance(vector, r["vec"]), r["key"]) for r in rows)
        return [key for _, key in scored[:limit]]


class StoreA:
    """Adapter: our contract, implemented with VendorA."""
    def __init__(self):
        self.backend = VendorA()

    def add(self, records):
        self.backend.upsert([
            {"id": r["id"], "values": embed(r["text"]),
             "metadata": {k: v for k, v in r.items() if k != "text"}}
            for r in records])

    def query(self, text, top_k=3, where=None):
        return self.backend.knn(embed(text), top_k,
                                filter={f: {"$eq": v} for f, v in (where or {}).items()}
                                or None)

    def count(self):
        return len(self.backend.items)


class StoreB:
    """Adapter: the SAME contract, implemented with VendorB. Your turn."""
    def __init__(self):
        self.backend = VendorB()

    def add(self, records):
        self.backend.insert([
            {"key": r["id"], "vec": embed(r["text"]),
             "meta": {k: v for k, v in r.items() if k != "text"}}
            for r in records])

    def query(self, text, top_k=3, where=None):
        # Every vendor's filter dialect is translated HERE and nowhere else. Several
        # fields would need an "and" expression; keeping it in one place is what makes
        # that a ten-line change rather than a hunt through the application.
        expression = None
        if where:
            field, value = next(iter(where.items()))
            expression = f"{field} == '{value}'"
        return self.backend.nearest(embed(text), top_k, expression)

    def count(self):
        return len(self.backend.rows)


def run_contract_tests(store):
    """The same tests, against any store that honours the contract."""
    store.add(RECORDS)
    results = {}
    results["count"] = store.count()
    results["unfiltered"] = store.query("return an item", top_k=3)
    results["filtered"] = store.query("return an item", top_k=3,
                                      where={"team": "support"})
    results["hr_only"] = store.query("return an item", top_k=3, where={"team": "hr"})
    return results


a, b = run_contract_tests(StoreA()), run_contract_tests(StoreB())
for key in a:
    same = "same" if a[key] == b[key] else "DIFFERENT"
    print(f"{key:>12}: {same:>9}  A={a[key]}  B={b[key]}")

In [ ]:
@check("9.2  both stores behave identically through the contract")
def _():
    need("StoreA", "StoreB", "run_contract_tests")
    a = run_contract_tests(StoreA())
    b = run_contract_tests(StoreB())
    assert a["count"] == b["count"] == len(RECORDS), \
        f"counts differ: A={a['count']}, B={b['count']}, expected {len(RECORDS)}"
    for key in ("unfiltered", "filtered", "hr_only"):
        assert a[key] == b[key], (
            f"{key}: StoreA returned {a[key]} and StoreB returned {b[key]}. Running the "
            "same tests against both stores is the point of the contract — a difference "
            "here is exactly the vendor behaviour change you want to catch."
        )
    print(f"     same results for all 3 queries: {a['filtered']}")


@check("9.2  the filter is translated, not ignored")
def _():
    need("StoreB")
    store = StoreB()
    store.add(RECORDS)

    hr = store.query("anything at all", top_k=5, where={"team": "hr"})
    assert set(hr) <= {"d4", "d5"}, (
        f"filtering on team=hr returned {hr}. Only d4 and d5 are HR documents — if "
        "other ids appear, the filter was dropped on the way to the vendor."
    )
    assert hr, "filtering on team=hr should return the two HR documents, not nothing"

    confidential = store.query("anything", top_k=5, where={"visibility": "confidential"})
    assert set(confidential) <= {"d4", "d5", "d7"}, \
        f"filtering on visibility=confidential returned {confidential}"

    everything = store.query("anything", top_k=8)
    assert len(everything) == len(RECORDS), \
        f"an unfiltered query should see all {len(RECORDS)} documents, got {len(everything)}"
    print("     filters translated correctly for 2 different fields")

In [ ]:
hint('add():',
     '  self.backend.insert([',
     "      {'key': r['id'], 'vec': embed(r['text']),",
     "       'meta': {k: v for k, v in r.items() if k != 'text'}}",
     '      for r in records])',
     '',
     'query():',
     '  expression = None',
     '  if where:',
     '      field, value = next(iter(where.items()))',
     '      expression = f"{field} == \'{value}\'"',
     '  return self.backend.nearest(embed(text), top_k, expression)',
     '',
     'count(): return len(self.backend.rows)')

**What to notice.** Two completely different vendor APIs, one set of tests, identical
results. Swapping database now means writing one adapter — not editing your application.

**Be honest about what a fake proves, though.** These two "vendors" are dictionaries. They
cannot show you a real database's latency, its index settings, its consistency delays, or
the ways it fails under load. A fake proves your *contract* is sound. Only the real thing
proves the *store* is.

---
## Exercise 3 — A filter that leaks  ·  *book §9.3*


This exercise is a bug hunt, and the bug is one that ships.

You have hybrid search from Chapter 8: a keyword path and a meaning path, fused. You also
have permissions — HR documents must not reach a support agent.

So someone adds the permission filter. To one of the two paths.

> **A filter that applies to one search path and not the other is not a filter.**

The function below has exactly that flaw. A support user searching for "salary" will see an
HR document, because the keyword path never checked. And it will not look like a security
hole — it will look like a relevant result.

The fix is an order-of-operations change, and it is worth saying out loud:

**Decide what this person may see *before* you rank anything.**

Not after. Not per path. Once, at the start, on the candidate set that every path then
draws from.

In [ ]:
VISIBLE_TO = {
    "support_agent": {"public", "internal"},
    "hr_manager":    {"public", "internal", "confidential"},
    "guest":         {"public"},
}


def keyword_hits(query, records):
    """Record ids sharing at least one word with the query, best first."""
    words = set(_words(query))
    scored = [(len(words & set(_words(r["text"]))), r["id"]) for r in records]
    return [doc_id for score, doc_id in sorted(scored, key=lambda p: -p[0]) if score > 0]


def meaning_hits(query, records):
    """Record ids by embedding distance, closest first."""
    q = embed(query)
    return [doc_id for _, doc_id in
            sorted((distance(q, VECTORS[r["id"]]), r["id"]) for r in records)]


def fuse(ranked_lists, k=60, top_k=3):
    """Reciprocal rank fusion, as in Chapter 8. Given to you."""
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k + rank)
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda p: -p[1])[:top_k]]


def leaky_search(query, role, top_k=3):
    """The version with the bug. Read it, then run the check to see it leak."""
    allowed = VISIBLE_TO[role]
    by_keyword = keyword_hits(query, RECORDS)                      # never filtered!
    by_meaning = meaning_hits(query, [r for r in RECORDS
                                      if r["visibility"] in allowed])
    return fuse([by_keyword, by_meaning], top_k=top_k)


def safe_search(query, role, top_k=3):
    """Your version. Decide what this role may see BEFORE ranking anything."""
    allowed = VISIBLE_TO[role]

    # ONE candidate set, decided before any ranking happens. Both paths draw from it,
    # so neither can surface something this role may not see.
    candidates = [r for r in RECORDS if r["visibility"] in allowed]

    by_keyword = keyword_hits(query, candidates)
    by_meaning = meaning_hits(query, candidates)
    return fuse([by_keyword, by_meaning], top_k=top_k)


print("query: 'salary bands for the support team'\n")
for role in ("support_agent", "hr_manager", "guest"):
    leaky = leaky_search("salary bands for the support team", role)
    safe = safe_search("salary bands for the support team", role)
    print(f"{role:>13}  leaky={leaky}  safe={safe}")

In [ ]:
@check("9.3  the leak is real (this check proves the bug exists)")
def _():
    need("leaky_search")
    leaked = leaky_search("salary bands for the support team", "support_agent", top_k=3)
    assert "d4" in leaked, (
        "the leaky version did not leak here, which means the exercise cannot "
        "demonstrate its point — check you have not already fixed leaky_search."
    )
    print(f"     leaky_search let a support agent see {leaked} — d4 is HR confidential")


@check("9.3  no role ever sees beyond its permissions")
def _():
    need("safe_search", "VISIBLE_TO")
    by_id = {r["id"]: r for r in RECORDS}

    probes = ["salary bands for the support team",
              "disciplinary process dismissal",
              "payment gateway credentials rotate",
              "refund abuse flagged accounts",
              "return an item for a refund"]

    for role, allowed in VISIBLE_TO.items():
        for query in probes:
            for doc_id in safe_search(query, role, top_k=8):
                visibility = by_id[doc_id]["visibility"]
                assert visibility in allowed, (
                    f"role {role!r} searching {query!r} received {doc_id} "
                    f"({visibility}), which it may not see. Filter the candidate set "
                    "ONCE, before ranking, and use it for both paths."
                )

    hr = safe_search("salary bands", "hr_manager", top_k=3)
    assert "d4" in hr, (
        f"an HR manager searching for salary bands got {hr} — the fix must not break "
        "legitimate access. Over-filtering is a bug too."
    )
    support = safe_search("return an item for a refund", "support_agent", top_k=3)
    assert "d1" in support, \
        f"a support agent should still find the returns policy; got {support}"
    print("     3 roles x 5 queries: no leak, and legitimate results still arrive")

In [ ]:
hint('safe_search is three lines:',
     '  allowed = VISIBLE_TO[role]',
     "  candidates = [r for r in RECORDS if r['visibility'] in allowed]",
     '  return fuse([keyword_hits(query, candidates),',
     '               meaning_hits(query, candidates)], top_k=top_k)',
     '',
     'Compare with leaky_search: it passes RECORDS to one path and a filtered list',
     'to the other. The fix is to build the candidate set once, first.')

**What to notice.** The leaky version was not careless-looking code. Someone added a filter,
tested it, saw it work — on the path they were looking at.

Three habits come out of this:

* **Filter once, before ranking.** One candidate set, every path draws from it.
* **Test access control by trying to break it.** The passing check above probes three roles
  against five queries. It is cheap and it is the only kind of test that finds this.
* **Over-filtering is also a bug.** The check insists HR managers still find HR documents.

---
## What you built

* An approximate index trades accuracy for speed. **Recall** measures what you lost, and nothing warns you without it.
* Measure recall against exact search on **your own** data and queries — it depends on both.
* Put every store behind **one small contract**, translate each vendor's filter dialect in one place, and run the same tests against all of them.
* A fake store proves your contract, not the database. Latency, index settings and failure under load need the real thing.
* **Decide what a person may see before you rank anything.** A filter on one search path and not another is not a filter.
* Test access control by **trying to break it** — and check that legitimate access still works.

**Next:** Chapter 10 goes one level down, to the embeddings themselves: comparing models, shrinking vectors, and caching them safely.

In [ ]:
progress()